# PR0301 · Exploración e ingesta en la Capa Bronce

**Módulo:** Big Data Aplicado  
**Unidad de Trabajo:** UT03. Almacenamiento distribuido en cloud con S3. Data Lakes  
**Práctica:** PR0301. Ingesta en Data Lake (Capa Bronce) sobre Amazon S3

Este cuaderno recoge la **Fase 2** de la práctica: la exploración y el prototipado
previos al paso a producción. Se comprueban las codificaciones y delimitadores de
los dos catálogos, el cruce por código INE, la API de AEMET (doble salto) y la
subida a S3 con metadatos de trazabilidad.

> **Requisitos:** laboratorio de AWS Academy iniciado y fichero `.env` actualizado
> con las credenciales temporales y la API Key de AEMET.


## 0. Entorno y variables de configuración

Cargamos las variables desde el `.env` de la raíz del proyecto. Por seguridad,
solo mostramos si cada variable está presente, nunca su valor.


In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import boto3
import pandas as pd
import requests
from dotenv import load_dotenv

# El cuaderno puede ejecutarse desde notebooks/ o desde la raíz del proyecto
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(BASE_DIR / ".env", override=True)

DATA_DIR = BASE_DIR / "data"
RUTA_MITECO = DATA_DIR / "playas.csv"
RUTA_AEMET = DATA_DIR / "playas_codigos.csv"

print("Raíz del proyecto:", BASE_DIR)
print("Variables cargadas:")
for nombre in [
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "AWS_SESSION_TOKEN",
    "AWS_DEFAULT_REGION",
    "S3_BUCKET_NAME",
    "AEMET_API_KEY",
]:
    print(f"  {nombre}: {'OK' if os.getenv(nombre) else 'FALTA'}")


## 1. Inspección de los catálogos

Los dos proveedores no comparten ni codificación ni separador:

| Fichero | Separador | Codificación | Filas |
|---|---|---|---|
| `playas.csv` (MITECO) | `,` | UTF-8 con BOM | 3.554 |
| `playas_codigos.csv` (AEMET) | `;` | ISO-8859-1 / cp1252 | 591 |

La función siguiente prueba varias codificaciones y se queda con la primera que
permite leer el fichero completo sin errores de decodificación.


In [ ]:
def leer_csv(ruta, separador):
    """Lee un CSV probando codificaciones habituales en estos catálogos."""
    for encoding in ("utf-8-sig", "ISO-8859-1", "cp1252"):
        try:
            df = pd.read_csv(ruta, sep=separador, encoding=encoding, dtype=str)
            return df, encoding
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"No se pudo leer {ruta} con las codificaciones probadas")


df_miteco, encoding_miteco = leer_csv(RUTA_MITECO, ",")
df_aemet, encoding_aemet = leer_csv(RUTA_AEMET, ";")

print(f"MITECO  -> {len(df_miteco):>4} filas | codificación detectada: {encoding_miteco}")
print(f"AEMET   -> {len(df_aemet):>4} filas | codificación detectada: {encoding_aemet}")
print()
print("Columnas MITECO:", list(df_miteco.columns[:15]), "...")
print("Columnas AEMET :", list(df_aemet.columns))


In [ ]:
# Primeras filas de cada catálogo (columnas más relevantes)
columnas_miteco = [
    "Identifica", "Código_IN", "Nombre", "Provincia", "Término_M",
    "Duchas", "Aseos", "Acceso_dis", "Bandera_az", "Grado_ocup",
]
display(df_miteco[columnas_miteco].head(5))
display(df_aemet.head(5))


## 2. Cruce de catálogos por código INE

El nexo de unión es el código INE del municipio:

- En MITECO: `Código_IN` (5 dígitos).
- En AEMET: `ID_MUNICIPIO` (5 dígitos).

`ID_PLAYA` es el identificador de 7 dígitos que necesita la API de AEMET y
**no** sirve para cruzar con MITECO. Normalizamos ambos códigos con `zfill`
para no perder los ceros a la izquierda.


In [ ]:
PROVINCIA = "Alicante"
MAX_PLAYAS = 15

# Normalización de códigos
df_miteco["Código_IN"] = df_miteco["Código_IN"].str.strip().str.zfill(5)
df_aemet["ID_MUNICIPIO"] = df_aemet["ID_MUNICIPIO"].str.strip().str.zfill(5)
df_aemet["ID_PLAYA"] = df_aemet["ID_PLAYA"].str.strip().str.zfill(7)

# Municipios de la provincia elegida en el catálogo MITECO
df_provincia = df_miteco[
    df_miteco["Provincia"].str.contains(PROVINCIA, case=False, na=False, regex=False)
]
codigos_ine = sorted(df_provincia["Código_IN"].dropna().unique())

print(f"Municipios de {PROVINCIA}: {len(codigos_ine)}")
print(codigos_ine)


In [ ]:
# Playas de AEMET pertenecientes a esos municipios
df_playas = (
    df_aemet[df_aemet["ID_MUNICIPIO"].isin(codigos_ine)]
    .drop_duplicates(subset=["ID_PLAYA"])
    .head(MAX_PLAYAS)
    .reset_index(drop=True)
)

print(f"Playas AEMET seleccionadas: {len(df_playas)}")
df_playas[["ID_PLAYA", "NOMBRE_PLAYA", "ID_MUNICIPIO", "NOMBRE_MUNICIPIO"]]


## 3. Prueba de la API de AEMET (doble salto)

AEMET OpenData funciona en dos pasos:

1. `GET /prediccion/especifica/playa/{ID_PLAYA}` con la cabecera `api_key`.
   Devuelve un JSON de control con el campo `datos` (URL temporal).
2. `GET` a esa URL temporal para descargar la predicción real.

Además de comprobar los códigos HTTP, hay que revisar el campo interno
`estado` del primer JSON. La pausa de 1,5 s evita el error 429 (rate limiting).


In [ ]:
AEMET_API_KEY = os.getenv("AEMET_API_KEY")
ID_PRUEBA = df_playas.iloc[0]["ID_PLAYA"]


def decodificar_json(respuesta):
    """Decodifica el JSON conservando tildes y eñes."""
    for encoding in ("utf-8-sig", "utf-8", "ISO-8859-15", "cp1252"):
        try:
            return json.loads(respuesta.content.decode(encoding))
        except (UnicodeDecodeError, json.JSONDecodeError):
            continue
    return respuesta.json()


def descargar_prediccion(id_playa):
    url = (
        "https://opendata.aemet.es/opendata/api/"
        f"prediccion/especifica/playa/{id_playa}"
    )
    headers = {"api_key": AEMET_API_KEY, "Accept": "application/json"}

    # Paso 1: JSON de control
    respuesta_control = requests.get(url, headers=headers, timeout=10)
    print("Paso 1 - HTTP:", respuesta_control.status_code)
    respuesta_control.raise_for_status()
    control = respuesta_control.json()
    print("Estado interno AEMET:", control.get("estado"))

    if str(control.get("estado")) != "200":
        print("AEMET no devolvió datos para esta playa.")
        return None

    # Paso 2: descarga del JSON real
    respuesta_datos = requests.get(control["datos"], timeout=10)
    print("Paso 2 - HTTP:", respuesta_datos.status_code)
    respuesta_datos.raise_for_status()
    return decodificar_json(respuesta_datos)


print(f"Probando la playa {ID_PRUEBA} ({df_playas.iloc[0]['NOMBRE_PLAYA']})...")
prediccion = descargar_prediccion(ID_PRUEBA)

if prediccion is not None:
    print("Tipo de objeto:", type(prediccion).__name__)
    print(json.dumps(prediccion, ensure_ascii=False, indent=2)[:1500])
    ruta_json = DATA_DIR / f"pred_{ID_PRUEBA}.json"
    ruta_json.write_text(
        json.dumps(prediccion, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("\nJSON guardado en:", ruta_json)

time.sleep(1.5)  # Rate limiting de AEMET


## 4. Prueba de subida a Amazon S3

Probamos las dos modalidades que exige la práctica:

- `upload_file`: subir un fichero que ya existe en disco.
- `put_object`: subir un JSON generado en memoria RAM.

En ambos casos se adjuntan los metadatos `source` e `ingestion_timestamp`.


In [ ]:
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
BUCKET = os.getenv("S3_BUCKET_NAME")

s3 = boto3.client("s3", region_name=AWS_REGION)


def crear_bucket_si_no_existe(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        print("El bucket ya existe:", bucket)
    except Exception as error:
        codigo = getattr(error, "response", {}).get("Error", {}).get("Code", "")
        if codigo in ("404", "NoSuchBucket", "NotFound"):
            argumentos = {"Bucket": bucket}
            if AWS_REGION != "us-east-1":
                argumentos["CreateBucketConfiguration"] = {
                    "LocationConstraint": AWS_REGION
                }
            s3.create_bucket(**argumentos)
            print("Bucket creado:", bucket)
        elif codigo in ("403", "AccessDenied"):
            raise PermissionError(
                f"El bucket {bucket} existe en otra cuenta o no tienes permisos."
            ) from error
        elif codigo in ("301", "PermanentRedirect"):
            print("El bucket existe en otra región; se usará tal cual.")
        else:
            raise


crear_bucket_si_no_existe(BUCKET)
print("Buckets de la cuenta:", [b["Name"] for b in s3.list_buckets()["Buckets"]])


In [ ]:
from datetime import datetime, timezone

ahora = datetime.now(timezone.utc)
timestamp_iso = ahora.strftime("%Y-%m-%dT%H:%M:%SZ")
particion = f"year={ahora.year}/month={ahora.month:02d}/day={ahora.day:02d}"

# CASO A: subida de un fichero local con upload_file
key_csv = "bronce/catalogos/miteco_playas/v1/playas.csv"
s3.upload_file(
    Filename=str(RUTA_MITECO),
    Bucket=BUCKET,
    Key=key_csv,
    ExtraArgs={
        "ContentType": "text/csv",
        "Metadata": {
            "source": "MITECO_OpenData",
            "ingestion_timestamp": timestamp_iso,
        },
    },
)

cabecera_csv = s3.head_object(Bucket=BUCKET, Key=key_csv)
print("Subido:", key_csv)
print("Metadatos:", cabecera_csv["Metadata"])


In [ ]:
# CASO B: subida de un JSON generado en memoria con put_object
if prediccion is None:
    print("No hay predicción para subir. Revisa la celda de AEMET.")
else:
    key_json = (
        "bronce/meteorologia/aemet_prediccion_playas/"
        f"{particion}/pred_playa_{ID_PRUEBA}_{ahora.strftime('%H%M%S')}.json"
    )
    cuerpo = json.dumps(prediccion, ensure_ascii=False, indent=2).encode("utf-8")

    s3.put_object(
        Bucket=BUCKET,
        Key=key_json,
        Body=cuerpo,
        ContentType="application/json",
        Metadata={
            "source": "AEMET_API",
            "ingestion_timestamp": timestamp_iso,
        },
    )

    cabecera_json = s3.head_object(Bucket=BUCKET, Key=key_json)
    print("Subido:", key_json)
    print("Metadatos:", cabecera_json["Metadata"])
    print("Tamaño:", cabecera_json["ContentLength"], "bytes")


## 5. Conclusiones de la exploración

- Los catálogos se leen correctamente si se respeta su **codificación** y su
  **separador**: MITECO en UTF-8 con BOM y comas; AEMET en ISO-8859-1 y punto y coma.
- El cruce fiable es `Código_IN` (MITECO) ↔ `ID_MUNICIPIO` (AEMET), no `ID_PLAYA`.
- Para la provincia de Alicante se obtienen 20 municipios y 31 playas en AEMET;
  el pipeline se queda con las 15 primeras.
- La API de AEMET exige dos peticiones y una pausa entre llamadas; el JSON final
  conserva correctamente tildes y eñes.
- `upload_file` y `put_object` permiten cubrir tanto ficheros de disco como datos
  generados en memoria, adjuntando siempre los metadatos de trazabilidad.

En la **Fase 3**, esta lógica se encapsula en `src/config.py`, `src/s3_manager.py`,
`src/aemet_client.py` y `src/main.py`.
